<a href="https://colab.research.google.com/github/laosrb/AI-Final-Project/blob/main/AI_Project_Social_Media.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
# IMPORT

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer

import lightgbm as lgb
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Load Datasets

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load Instagram analytics dataset (12,000)
instagram_data = pd.read_csv("/content/drive/MyDrive/AI Final Project Ryan Bouapheng/Instagram_Analytics.csv")

# Load social media engagement dataset (30,000)
social_engagement = pd.read_csv("/content/drive/MyDrive/AI Final Project Ryan Bouapheng/Social Media Engagement Dataset.csv")

# Preview data
print(instagram_data.head())
print(social_engagement.head())

     post_id  account_id account_type  follower_count media_type  \
0  IG0000001           7        brand            3551       reel   
1  IG0000002          20      creator           31095      image   
2  IG0000003          15        brand            8167       reel   
3  IG0000004          11      creator            9044   carousel   
4  IG0000005           8      creator           15986       reel   

  content_category traffic_source  has_call_to_action        post_datetime  \
0       Technology      Home Feed                   1  2024-11-30 06:00:00   
1          Fitness       Hashtags                   1  2025-08-15 15:00:00   
2           Beauty     Reels Feed                   0  2025-09-11 16:00:00   
3            Music       External                   0  2025-09-18 03:00:00   
4       Technology        Profile                   0  2025-03-21 09:00:00   

    post_date  ...  comments shares  saves  reach  impressions  \
0  2024-11-30  ...         5      7     34   4327       

# Data Preprocessing

In [4]:
# Clean captions
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    # Keep alphanumeric, spaces, and hashtags
    text = ''.join([c for c in text if c.isalnum() or c.isspace() or c == '#'])
    return text

# Process social_engagement text
if 'text_content' in social_engagement.columns:
    social_engagement['clean_caption'] = social_engagement['text_content'].apply(clean_text)

# Handle the length mismatch and indexing error
captions_subset = social_engagement['text_content'].iloc[:len(instagram_data)].reset_index(drop=True)
instagram_data['clean_caption'] = captions_subset.apply(clean_text)

# Handle missing values for engagement metrics
instagram_data.fillna({'likes': 0, 'comments': 0, 'shares': 0}, inplace=True)

# Extract hour from 'post_datetime'
instagram_data['posting_hour'] = pd.to_datetime(instagram_data['post_datetime']).dt.hour

# One-hot encode media_type (Reel, Image, Carousel)
if 'media_type' in instagram_data.columns:
    instagram_data = pd.get_dummies(instagram_data, columns=['media_type'], drop_first=True)

print("Preprocessing complete!")
print(f"Dataset shape: {instagram_data.shape}")

Preprocessing complete!
Dataset shape: (29999, 26)


# Feature Engineering

In [7]:
# Replace NaN values in clean_caption with an empty string to prevent the TF-IDF error
instagram_data['clean_caption'] = instagram_data['clean_caption'].fillna('')

# TF-IDF for captions (Green AI: limited to 500 features)
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words=stopwords.words('english'))
caption_tfidf = tfidf_vectorizer.fit_transform(instagram_data['clean_caption']).toarray()

# Combine with numeric features
numeric_features = instagram_data[['posting_hour', 'follower_count']].values

# Stack features together
X = np.hstack((caption_tfidf, numeric_features))
y = instagram_data['likes'].values

print(f"Feature matrix X shape: {X.shape}")
print("Feature engineering complete!")

Feature matrix X shape: (29999, 283)
Feature engineering complete!


# Split Training & Testing Data

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Efficient Model

In [9]:
# LightGBM model
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train, y_train)
y_pred = lgb_model.predict(X_test)

# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R2 Score: {r2}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.069225 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31244
[LightGBM] [Info] Number of data points in the train set: 23999, number of used features: 283
[LightGBM] [Info] Start training from score 277.063919


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Mean Squared Error: 203763.0387342559
R2 Score: -0.026639702206052496


# Optimize Algorithms

In [10]:
def predict_engagement(caption, posting_hour, followers_count):
    caption_clean = clean_text(caption)
    caption_vec = tfidf_vectorizer.transform([caption_clean]).toarray()
    features = np.hstack((caption_vec, np.array([[posting_hour, followers_count]])))
    predicted_likes = lgb_model.predict(features)[0]
    return predicted_likes

# Example usage
caption = "Check out my new summer collection! #fashion #style"
predicted_likes = predict_engagement(caption, posting_hour=15, followers_count=1200)
print(f"Predicted Likes: {int(predicted_likes)}")

Predicted Likes: 231


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Green AI Strategies for Your Project

## Use Lightweight Models
- Avoid large neural networks.
- Use models like:
  - LightGBM
  - XGBoost
  - Random Forests with limited depth

## Limit Feature Dimensions
- Reduce TF-IDF vector size (e.g., 500–1000 features instead of thousands).
- Drop low-importance numeric and categorical features.

## Early Stopping
- Stop training when performance stops improving to save computation.

## Batch Processing
- Predict in batches instead of per-post to reduce repeated computations.

## Energy Tracking (Optional)
- Use the `time` module to track training duration as a proxy for energy usage.

## Reproducibility
- Fix random seeds to avoid repeated runs for the same results.

# Preprocessing


In [12]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    # Keep alphanumeric, spaces, and hashtags
    text = ''.join([c for c in text if c.isalnum() or c.isspace() or c == '#'])
    return text

# grab the text from the social_engagement dataset and align it to instagram_data
if 'text_content' in social_engagement.columns:
    # Reset index ensures the rows line up perfectly without index errors
    captions_subset = social_engagement['text_content'].iloc[:len(instagram_data)].reset_index(drop=True)
    instagram_data['clean_caption'] = captions_subset.apply(clean_text)
else:
    # Handle if text_content isn't found
    instagram_data['clean_caption'] = ""

# Handle missing values
instagram_data.fillna({'likes': 0, 'comments': 0, 'shares': 0}, inplace=True)

instagram_data['posting_hour'] = pd.to_datetime(instagram_data['post_datetime']).dt.hour

print("Preprocessing successful! Columns updated.")

Preprocessing successful! Columns updated.


# Feature Engineering (Green AI)

In [14]:
instagram_data['clean_caption'] = instagram_data['clean_caption'].fillna('')

# TF-IDF for captions (Green AI: limited features)
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words=stopwords.words('english'))
caption_tfidf = tfidf_vectorizer.fit_transform(instagram_data['clean_caption']).toarray()

# Combine with numeric features
numeric_features = instagram_data[['posting_hour', 'follower_count']].values

# Stack features together
X = np.hstack((caption_tfidf, numeric_features))
y = instagram_data['likes'].values

print(f"Feature matrix X shape: {X.shape}")
print("Feature engineering complete!")

Feature matrix X shape: (29999, 283)
Feature engineering complete!


# Train-Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Lightweight Green AI Model


In [17]:
import time
import lightgbm as lgb

# Initialize the model
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,       # smaller number of trees saves energy
    learning_rate=0.05,
    num_leaves=31,           # limited complexity
    random_state=42
)

# Track training time (proxy for energy consumption)
start_time = time.time()

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # stop early to save energy
        lgb.log_evaluation(period=0)            # period=0 hides the logs (replaces verbose=False)
    ]
)

end_time = time.time()

training_duration = end_time - start_time
print(f"Training Time (seconds): {training_duration:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.072196 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31244
[LightGBM] [Info] Number of data points in the train set: 23999, number of used features: 283
[LightGBM] [Info] Start training from score 277.063919
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[2]	valid_0's rmse: 448.624	valid_0's l2: 201264
Training Time (seconds): 1.05


# Model Evaluation


In [18]:
y_pred = lgb_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")



Mean Squared Error: 201263.65
R2 Score: -0.01


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Prediction Function (Batch-Friendly)


In [19]:
def predict_engagement(captions, posting_hours, followers_counts):
    # Predict engagement for multiple posts at once (Green AI: batch processing)
    captions_clean = [clean_text(c) for c in captions]
    captions_vec = tfidf_vectorizer.transform(captions_clean).toarray()
    numeric_features = np.array([posting_hours, followers_counts]).T
    features = np.hstack((captions_vec, numeric_features))
    return lgb_model.predict(features)

# Example usage
captions = ["Check out my new summer collection! #fashion #style",
            "Behind the scenes of our latest photoshoot!"]
posting_hours = [15, 18]
followers_counts = [1200, 1500]

predicted_likes = predict_engagement(captions, posting_hours, followers_counts)
print("Predicted Likes:", predicted_likes)

Predicted Likes: [276.93326012 276.93326012]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Enhance Prediction & Scoring Logic
We need to ensure the model produces a score. We'll normalize the predicted likes against your historical maximum to get that (0-100) range.

In [20]:
import numpy as np

def get_performance_score(predicted_likes, max_historical_likes):
    """Converts raw like prediction to a 0-100 scale."""
    score = (predicted_likes / max_historical_likes) * 100
    return min(100, round(score, 2))

# Calculate this once after loading data
max_likes = instagram_data['likes'].max()

# Optimizer Algorithm
This function doesn't just predict; it iterates through different scenarios (like every hour of the day) to find the "peak" performance for your specific caption.

In [21]:
def optimize_post(caption, followers_count):
    best_score = -1
    best_hour = 0

    # 1. Analyze the current caption
    # (Extract hashtags if they exist in the string)
    has_hashtags = "#" in caption

    # 2. Iterative Optimization Loop (Testing every hour of the day)
    for hour in range(24):
        # Clean and Transform
        caption_clean = clean_text(caption)
        caption_vec = tfidf_vectorizer.transform([caption_clean]).toarray()
        features = np.hstack((caption_vec, np.array([[hour, followers_count]])))

        # Predict
        prediction = lgb_model.predict(features)[0]
        score = get_performance_score(prediction, max_likes)

        if score > best_score:
            best_score = score
            best_hour = hour

    # 3. Generate Human-Readable Suggestions
    suggestions = []
    if not has_hashtags:
        suggestions.append("Strategy: Adding 3-5 relevant hashtags could increase reach.")

    if len(caption) < 20:
        suggestions.append("Strategy: Your caption is very short. Try adding a Call to Action (CTA).")

    return {
        "Predicted Performance Score": f"{best_score}/100",
        "Recommended Posting Time": f"{best_hour}:00",
        "Optimization Tips": suggestions
    }

# Use Case

In [22]:
new_post = "Launching my new project today! #tech #innovation"
analysis = optimize_post(new_post, followers_count=1500)

print(f"Post Score: {analysis['Predicted Performance Score']}")
print(f"Best Time to Post: {analysis['Recommended Posting Time']}")
for tip in analysis['Optimization Tips']:
    print(tip)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Post Score: 2.6/100
Best Time to Post: 0:00


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v